# Week 04 : NLP Architectures

### 0. Setup

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import matplotlib.pyplot as plt
import time, random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ===== 실습용 빠른 실행 설정 =====
# 너무 오래 걸리면 아래 값만 먼저 줄여서 실행하세요.
FAST_RUN = True
TRAIN_LIMIT = 4000 if FAST_RUN else None
TEST_LIMIT  = 1000 if FAST_RUN else None
MAX_SEQ     = 128 if FAST_RUN else 256
TRAIN_BATCH_SIZE = 64
TEST_BATCH_SIZE  = 128
EPOCHS = 3 if FAST_RUN else 3

# ── 공통 학습/평가 함수 ───────────────────────────
def train_model(model, loader, optimizer, criterion, epochs=EPOCHS, label=''):
    model.train()
    loss_hist = []
    for epoch in range(epochs):
        total = 0
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(texts), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item()
        avg = total / len(loader)
        loss_hist.append(avg)
        print(f'  [{label}] Epoch {epoch+1}/{epochs}  loss={avg:.4f}')
    return loss_hist

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            preds = model(texts).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 최종 비교용 결과 저장
results = {}

Using device: cuda


In [7]:
from pathlib import Path
from collections import Counter
import os, re, tarfile, urllib.request

IMDB_URL    = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
IMDB_ROOT   = Path("./data_imdb")
EXTRACT_DIR = IMDB_ROOT / "aclImdb"
ARCHIVE_PATH = IMDB_ROOT / "aclImdb_v1.tar.gz"


class SimpleVocab:
    def __init__(self, stoi):
        self.stoi = stoi
        self.itos = [None] * len(stoi)
        for token, idx in stoi.items():
            self.itos[idx] = token
        self.default_index = 0

    def __len__(self):
        return len(self.stoi)

    def __getitem__(self, token):
        return self.stoi[token]

    def __call__(self, tokens):
        return [self.stoi.get(t, self.default_index) for t in tokens]

    def set_default_index(self, index):
        self.default_index = index


def basic_english_tokenizer(text):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"[^a-z0-9'.,!?;()\-]", " ", text)
    text = re.sub(r"([.,!?;()\-])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()


def download_and_extract_imdb():
    IMDB_ROOT.mkdir(parents=True, exist_ok=True)
    if not EXTRACT_DIR.exists():
        if not ARCHIVE_PATH.exists():
            print("Downloading IMDB dataset (~84 MB)...")
            urllib.request.urlretrieve(IMDB_URL, ARCHIVE_PATH)
            print("Download complete.")
        print("Extracting IMDB dataset...")
        with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
            tar.extractall(IMDB_ROOT)
        print("Extraction complete.")
    return EXTRACT_DIR


def load_imdb_split(split):
    """Returns list of (label_str, text) tuples. label_str is 'pos' or 'neg'."""
    root = download_and_extract_imdb() / split
    data = []
    for label in ["neg", "pos"]:
        label_dir = root / label
        for filename in sorted(os.listdir(label_dir)):
            if not filename.endswith(".txt"):
                continue
            with open(label_dir / filename, "r", encoding="utf-8") as f:
                data.append((label, f.read().strip()))
    return data


def build_vocab(texts, tokenizer, max_tokens=10_000, specials=("<unk>", "<pad>")):
    counter = Counter()
    for text in texts:
        counter.update(tokenizer(text))
    stoi = {}
    for token in specials:
        if token not in stoi:
            stoi[token] = len(stoi)
    keep_n = max_tokens - len(stoi)
    for token, _ in counter.most_common(keep_n):
        if token not in stoi:
            stoi[token] = len(stoi)
    vocab = SimpleVocab(stoi)
    vocab.set_default_index(vocab["<unk>"])
    return vocab


def maybe_limit_dataset(data, limit=None, seed=42):
    if limit is None or limit >= len(data):
        return data
    rng = random.Random(seed)
    idxs = list(range(len(data)))
    rng.shuffle(idxs)
    idxs = idxs[:limit]
    return [data[i] for i in idxs]


tokenizer = basic_english_tokenizer

print("Loading IMDB train split...")
train_data = load_imdb_split("train")
print("Loading IMDB test split...")
test_data  = load_imdb_split("test")

train_data = maybe_limit_dataset(train_data, TRAIN_LIMIT, seed=42)
test_data  = maybe_limit_dataset(test_data, TEST_LIMIT, seed=43)

print("Building vocabulary...")
vocab_imdb = build_vocab(
    [text for _, text in train_data],
    tokenizer=tokenizer,
    max_tokens=10_000,
    specials=["<unk>", "<pad>"],
)
print(f"Train samples: {len(train_data):,} | Test samples: {len(test_data):,}")
print(f"Vocab size   : {len(vocab_imdb):,}")
print(f"MAX_SEQ={MAX_SEQ}, EPOCHS={EPOCHS}, FAST_RUN={FAST_RUN}")

Loading IMDB train split...
Loading IMDB test split...
Building vocabulary...
Train samples: 4,000 | Test samples: 1,000
Vocab size   : 10,000
MAX_SEQ=128, EPOCHS=1, FAST_RUN=True


In [8]:
def text_pipeline(text):
    return vocab_imdb(tokenizer(text)[:MAX_SEQ])

def label_pipeline(label):
    return 1 if label == "pos" else 0

def collate_fn(batch):
    labels, texts = [], []
    for label, text in batch:
        labels.append(label_pipeline(label))
        texts.append(torch.tensor(text_pipeline(text), dtype=torch.long))
    texts_pad = pad_sequence(
        texts,
        batch_first=True,
        padding_value=vocab_imdb["<pad>"]
    )
    return texts_pad, torch.tensor(labels, dtype=torch.long)

train_loader_imdb = DataLoader(
    train_data, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
test_loader_imdb  = DataLoader(
    test_data, batch_size=TEST_BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

VOCAB_SIZE = len(vocab_imdb)
PAD_IDX    = vocab_imdb["<pad>"]
print(f"Train batches: {len(train_loader_imdb)}  |  Test batches: {len(test_loader_imdb)}")

Train batches: 63  |  Test batches: 8


## 1. MLP

### Task
- **문장 전체 감성 분류 (positive / negative)**
- 문장 안 단어들의 순서는 거의 무시하고, 임베딩들의 평균으로 전체 문장을 대표하게 만듭니다.
- 핵심 포인트:
  1. `Embedding`으로 각 단어를 벡터로 바꾸기
  2. **어느 축(dim)을 평균낼지** 이해하기
  3. 평균 벡터를 `Linear` 층으로 분류하기

> 아키텍처: `Embedding → Mean Pooling → Linear(ReLU) → Dropout → Linear`

```text
입력(x)        → (batch, seq_len)
Embedding      → (batch, seq_len, embed_dim)
Mean Pooling   → (batch, embed_dim)
FC1 + ReLU     → (batch, hidden_dim)
Dropout
FC2 (logits)   → (batch, num_classes)
```

In [9]:
class SentimentMLP(nn.Module):
    """
    MLP (BoW) 기반 감성 분류기
    - Embedding → Mean Pooling (순서 무시) → FC layers → 분류
    - 가장 단순한 딥러닝 NLP 베이스라인
    """
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_classes=2, dropout=0.5):
        super().__init__()

        # 문제 : 단어 id를 dense vector로 바꾸는 층은?
        # 정답 : nn.Embedding(vocab_size, embed_dim, padding_idx=1)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=1)

        # 문제 : 평균낸 문장 벡터를 hidden 벡터로 보내는 층은?
        # 정답 : nn.Linear(embed_dim, hidden_dim)
        self.fc1      = nn.Linear(embed_dim, hidden_dim)
        self.dropout  = nn.Dropout(dropout)

        # 문제 : 최종 2-class 분류층은?
        # 정답 : nn.Linear(hidden_dim, num_classes)
        self.fc2      = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)          # (batch, seq_len, embed_dim)

        # 문제 : 평균을 내야 하는 축(dim)은?
        # 정답 : 1
        pooled = emb.mean(dim = 1)

        out = torch.relu(self.fc1(pooled))
        out = self.dropout(out)
        return self.fc2(out)             # (batch, num_classes)

VOCAB_SIZE = len(vocab_imdb)
mlp_model = SentimentMLP(VOCAB_SIZE).to(device)
print(f"MLP 파라미터 수: {sum(p.numel() for p in mlp_model.parameters()):,}")

MLP 파라미터 수: 1,313,538


In [13]:
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

mlp_losses = train_model(
    model=mlp_model,
    loader=train_loader_imdb,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    label='MLP'
)
mlp_acc = evaluate(mlp_model, test_loader_imdb)
results['MLP'] = {'acc': mlp_acc,
                  'params': sum(p.numel() for p in mlp_model.parameters())}
print(f"\nMLP Test Accuracy: {mlp_acc:.4f}")

  [MLP] Epoch 1/3  loss=0.6270
  [MLP] Epoch 2/3  loss=0.5552
  [MLP] Epoch 3/3  loss=0.4763

MLP Test Accuracy: 0.7530


In [14]:
''' Task Cell : MLP shape tracing '''

mlp_model.eval()
batch_x, _ = next(iter(test_loader_imdb))
batch_x = batch_x.to(device)

with torch.no_grad():
    emb = mlp_model.embedding(batch_x)

    # 문제 : seq_len 축 평균
    # 정답 : 1
    pooled = emb.mean(dim = 1)

    # 문제 : 비선형 함수는?
    # 정답 : relu
    h1 = torch.relu(mlp_model.fc1(pooled))
    logits = mlp_model.fc2(mlp_model.dropout(h1))

print(f"입력 x       shape: {batch_x.shape}")      # (batch, seq_len)
print(f"Embedding   shape: {emb.shape}")          # (batch, seq_len, embed_dim)
print(f"MeanPool    shape: {pooled.shape}")       # (batch, embed_dim)
print(f"FC1+ReLU    shape: {h1.shape}")           # (batch, hidden_dim)
print(f"Logits      shape: {logits.shape}")       # (batch, num_classes)

입력 x       shape: torch.Size([128, 128])
Embedding   shape: torch.Size([128, 128, 128])
MeanPool    shape: torch.Size([128, 128])
FC1+ReLU    shape: torch.Size([128, 256])
Logits      shape: torch.Size([128, 2])


## 2. CNN

### Task
- **문장 내 지역적 패턴(local pattern) 감성 분류**
- 예: `"not good"`, `"very boring"`, `"really loved"` 같은 **짧은 n-gram 패턴**을 잘 잡는 모델입니다.
- 핵심 포인트:
  1. `Conv1d` 입력 shape가 왜 `(batch, channels, length)`인지
  2. 왜 `transpose(1, 2)`가 필요한지
  3. 시간축(문장 길이축)에서 `max pooling` 하는 이유

```text
Embedding → [Conv1d(k=3), Conv1d(k=5), Conv1d(k=7)] → MaxPool → Concat → FC
```

**핵심:** `nn.Conv1d`의 입력 shape는 `(batch, channels, length)`입니다.  
Embedding 출력 `(batch, seq_len, embed_dim)` → `.transpose(1, 2)` → `(batch, embed_dim, seq_len)`

In [15]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=64,
                 kernel_sizes=[3, 5, 7], num_classes=2, dropout=0.5):
        super().__init__()

        # 문제 : 단어 id -> embedding vector
        # 정답 : nn.Embedding(vocab_size, embed_dim, padding_idx=vocab_imdb['<pad>'])
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab_imdb['<pad>'])

        # 문제 : 서로 다른 kernel size의 Conv1d 여러 개 만들기
        # 정답 : nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        # 문제 : concat 이후 최종 분류층의 입력 차원은?
        # 정답 : num_filters * len(kernel_sizes)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x)               # (batch, seq, embed)

        # 문제 : Conv1d 입력 shape로 바꾸기 위해 transpose할 축은?
        # 정답 : 1, 2
        x = x.transpose(1, 2)               # (batch, embed, seq)

        # 문제 : max pooling은 어느 축(dim)으로?
        # 정답 : dim = 2 (문장 길이 축)
        pooled = [F.relu(conv(x)).max(dim = 2).values for conv in self.convs]

        x = torch.cat(pooled, dim=1)        # (batch, num_filters * 3)
        return self.fc(self.dropout(x))

In [16]:
# 모델 초기화 (IMDB vocab 사용)
EMBED_DIM    = 128
NUM_FILTERS  = 64
KERNEL_SIZES = [3, 5, 7]
NUM_CLASSES  = 2

cnn_model = TextCNN(VOCAB_SIZE, EMBED_DIM, NUM_FILTERS, KERNEL_SIZES, NUM_CLASSES).to(device)
print(cnn_model)
print(f'Parameters: {sum(p.numel() for p in cnn_model.parameters()):,}')

TextCNN(
  (embedding): Embedding(10000, 128, padding_idx=1)
  (convs): ModuleList(
    (0): Conv1d(128, 64, kernel_size=(3,), stride=(1,))
    (1): Conv1d(128, 64, kernel_size=(5,), stride=(1,))
    (2): Conv1d(128, 64, kernel_size=(7,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=192, out_features=2, bias=True)
)
Parameters: 1,403,458


In [17]:
# IMDB 학습
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3)

cnn_losses = train_model(
    cnn_model,
    train_loader_imdb,
    cnn_optimizer,
    cnn_criterion,
    epochs=EPOCHS,
    label='CNN'
)

cnn_acc = evaluate(cnn_model, test_loader_imdb)
results['CNN'] = {'acc': cnn_acc,
                  'params': sum(p.numel() for p in cnn_model.parameters())}

print(f'\nTextCNN Test Accuracy: {cnn_acc:.2%}')

  [CNN] Epoch 1/3  loss=0.7868
  [CNN] Epoch 2/3  loss=0.6688
  [CNN] Epoch 3/3  loss=0.5850

TextCNN Test Accuracy: 73.10%


In [18]:
''' Task Cell : CNN shape tracing '''

cnn_model.eval()
with torch.no_grad():
    sample_texts, _ = next(iter(test_loader_imdb))
    sample = sample_texts[:4].to(device)   # (4, MAX_SEQ)

    embedded = cnn_model.embedding(sample)    # (4, MAX_SEQ, embed_dim)
    print(f'After Embedding: {embedded.shape}')

    # 문제 : Conv1d를 위해 transpose할 축은?
    # 정답 : 1, 2
    transposed = embedded.transpose(1, 2)
    print(f'After Transpose: {transposed.shape}')

    # 문제 : max pooling은 어느 축(dim)으로?
    # 정답 : 2
    for i, (conv, k) in enumerate(zip(cnn_model.convs, KERNEL_SIZES)):
        conv_out = F.relu(conv(transposed))
        pooled   = conv_out.max(dim = 2).values
        print(f'  kernel={k}: conv_out={conv_out.shape}, pooled={pooled.shape}')

# Expected example
# After Embedding : torch.Size([4, 128, 128])  # FAST_RUN=True이면 seq 길이가 더 짧을 수 있음
# After Transpose : torch.Size([4, 128, 128])

After Embedding: torch.Size([4, 128, 128])
After Transpose: torch.Size([4, 128, 128])
  kernel=3: conv_out=torch.Size([4, 64, 126]), pooled=torch.Size([4, 64])
  kernel=5: conv_out=torch.Size([4, 64, 124]), pooled=torch.Size([4, 64])
  kernel=7: conv_out=torch.Size([4, 64, 122]), pooled=torch.Size([4, 64])


## 3. RNN

### Task
- **단어 순서를 따라가며 감성 분류**
- 앞에서 본 단어 정보를 뒤로 전달하면서 문장 의미를 누적합니다.
- 핵심 포인트:
  1. `input_size = embed_dim`
  2. `hidden_size`가 RNN의 기억 벡터 크기
  3. 마지막 hidden state `h_n[-1]`를 분류에 사용

```text
Embedding → RNN → last hidden state → FC → pos / neg
```

**힌트**: `nn.RNN(input_size, hidden_size, num_layers, batch_first, dropout)`

In [19]:
class SentimentRNN(nn.Module):
    """단방향 RNN 감성 분류기"""
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128,
                 num_layers=2, num_classes=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=vocab_imdb['<pad>']
        )

        # 문제 : RNN 입력 차원은? input_size=________            # 정답 : embed_dim
        # 문제 : hidden state 크기는? hidden_size=________      # 정답 : hidden_size
        # 문제 : 층 수는? num_layers=________                   # 정답 : num_layers
        self.rnn = nn.RNN(
            input_size = embed_dim,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        _, h_n = self.rnn(emb)  # h_n: (num_layers, batch, hidden)

        # 문제 : 마지막 레이어 hidden state 선택
        # 정답 : -1
        last_hidden = h_n[-1]

        return self.fc(self.dropout(last_hidden))


# 학습
rnn_model = SentimentRNN(VOCAB_SIZE).to(device)

rnn_losses = train_model(
    rnn_model,
    train_loader_imdb,
    optim.Adam(rnn_model.parameters(), lr=1e-3),
    nn.CrossEntropyLoss(),
    epochs=EPOCHS,
    label='RNN'
)

rnn_acc = evaluate(rnn_model, test_loader_imdb)
results['RNN'] = {'acc': rnn_acc,
                  'params': sum(p.numel() for p in rnn_model.parameters())}
print(f'\nRNN Test Accuracy:  {rnn_acc:.2%}')

  [RNN] Epoch 1/3  loss=0.7173
  [RNN] Epoch 2/3  loss=0.6933
  [RNN] Epoch 3/3  loss=0.6771

RNN Test Accuracy:  48.20%


## 4. LSTM

### Task
- **긴 문장에서 중요한 정보를 더 오래 기억하며 감성 분류**
- RNN보다 장기 의존성(long-term dependency)에 더 강합니다.
- 핵심 포인트:
  1. `nn.LSTM`은 hidden state와 cell state를 둘 다 반환
  2. 최종 분류에는 보통 `h_n[-1]` 사용
  3. `c_n`은 내부 기억 저장소 역할

```text
Embedding → LSTM → last hidden state → FC → pos / neg
```

In [20]:
class SentimentLSTM1(nn.Module):
    """단방향 LSTM 감성 분류기"""
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128,
                 num_layers=2, num_classes=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        # 문제 : 단어 임베딩 층
        # 정답 : nn.Embedding(vocab_size, embed_dim, padding_idx=vocab_imdb['<pad>'])
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab_imdb['<pad>'])

        # 문제 : LSTM 핵심 인자 (input_size=______, hidden_size=______, num_layers=______)
        # 정답 : embed_dim, hidden_size, num_layers
        self.lstm = nn.LSTM(
            input_size = embed_dim,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)          # (batch, seq, embed)

        # 문제 : LSTM 반환값에서 hidden, cell을 받기
        # 정답 : (h_n, c_n)
        a, (h_n, c_n) = self.lstm(emb)

        # 문제 : 마지막 레이어 hidden state 선택
        # 정답 : h_n[-1]
        return self.fc(self.dropout(h_n[-1]))

    def init_hidden(self, batch_size, device):
        """hidden / cell state shape 확인용"""
        # 문제 : LSTM은 h0, c0 두 개가 필요함
        # 정답 : 둘 다 (self.num_layers, batch_size, self.hidden_size)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)
        return (h0, c0)


lstm1_model = SentimentLSTM1(VOCAB_SIZE).to(device)
print(lstm1_model)
print(f'Parameters: {sum(p.numel() for p in lstm1_model.parameters()):,}')

SentimentLSTM1(
  (embedding): Embedding(10000, 128, padding_idx=1)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)
Parameters: 1,544,450


In [21]:
# IMDB 학습
lstm1_losses = train_model(
    lstm1_model,
    train_loader_imdb,
    optim.Adam(lstm1_model.parameters(), lr=1e-3),
    nn.CrossEntropyLoss(),
    epochs=EPOCHS,
    label='LSTM'
)

lstm1_acc = evaluate(lstm1_model, test_loader_imdb)
results['LSTM'] = {'acc': lstm1_acc,
                   'params': sum(p.numel() for p in lstm1_model.parameters())}
print(f'\nLSTM Test Accuracy: {lstm1_acc:.2%}')

# hidden / cell state shape 확인
sample_texts, _ = next(iter(test_loader_imdb))
sample_texts = sample_texts[:4].to(device)

with torch.no_grad():
    embedded = lstm1_model.embedding(sample_texts)
    output, (h_n, c_n) = lstm1_model.lstm(embedded)

print(f'Embedded shape : {embedded.shape}')  # (batch, seq_len, embed_dim)
print(f'Output shape   : {output.shape}')    # (batch, seq_len, hidden_size)
print(f'h_n shape      : {h_n.shape}')       # (num_layers, batch, hidden_size)
print(f'c_n shape      : {c_n.shape}')       # (num_layers, batch, hidden_size)

  [LSTM] Epoch 1/3  loss=0.6917
  [LSTM] Epoch 2/3  loss=0.6734
  [LSTM] Epoch 3/3  loss=0.6403

LSTM Test Accuracy: 53.60%
Embedded shape : torch.Size([4, 128, 128])
Output shape   : torch.Size([4, 128, 128])
h_n shape      : torch.Size([2, 4, 128])
c_n shape      : torch.Size([2, 4, 128])


### 체크 포인트
- **MLP**: 순서 무시, 가장 단순한 기준선
- **CNN**: 짧은 구간 패턴(n-gram) 포착
- **RNN**: 순서를 따라 누적
- **LSTM**: 더 오래 기억할 수 있도록 hidden + cell state 사용

### 생각해볼 질문
1. 감성 분류에서 `"not good"` 같은 표현은 왜 CNN/RNN/LSTM이 MLP보다 유리할까요?
2. 문장이 길어질수록 RNN보다 LSTM이 유리한 이유는 무엇일까요?
3. 속도는 느리지만 성능이 좋아질 수 있는 모델은 무엇일까요?

In [22]:
print(f'MLP  Test Acc: {results["MLP"]["acc"]:.2%}  |  Params: {results["MLP"]["params"]:,}')
print(f'CNN  Test Acc: {results["CNN"]["acc"]:.2%}  |  Params: {results["CNN"]["params"]:,}')
print(f'RNN  Test Acc: {rnn_acc:.2%}  |  Params: {results["RNN"]["params"]:,}')
print(f'LSTM Test Acc: {lstm1_acc:.2%}  |  Params: {results["LSTM"]["params"]:,}')
print(f'\n현재 설정: FAST_RUN={FAST_RUN}, EPOCHS={EPOCHS}, MAX_SEQ={MAX_SEQ}')

MLP  Test Acc: 75.30%  |  Params: 1,313,538
CNN  Test Acc: 73.10%  |  Params: 1,403,458
RNN  Test Acc: 48.20%  |  Params: 1,346,306
LSTM Test Acc: 53.60%  |  Params: 1,544,450

현재 설정: FAST_RUN=True, EPOCHS=3, MAX_SEQ=128
